# Haptic versus NoHA physiology by phase

This notebook compares participant-level physiological responses between Haptic and NoHA separately within each experimental phase. It uses participant × phase medians and two-sided Mann–Whitney U tests.

Interpretation is phase-specific:

- **pre_test:** group differences before training; no haptic feedback is active.
- **test_1–test_3:** physiological differences between Haptic and NoHA during training.
- **evaluation:** post-training physiological differences between groups; these are not effects of concurrent haptic stimulation.

The analysis does not test Group × Phase interactions and does not use MWL as a predictor or outcome.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import iqr, mannwhitneyu

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 190)

## 2. Configuration and paths

In [ ]:
FDR_ALPHA = 0.05
SAVE_FIGURES = True
SAVE_TABLES = True
TOP_N = 10
PHASE_ORDER = ["pre_test", "test_1", "test_2", "test_3", "evaluation"]
MODALITY_ORDER = ["ECG", "EDA", "RESP", "TEMP", "fNIRS"]

def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "outputs/final_features/physiology_mwl_analysis_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")

REPO_ROOT = find_repository_root()
INPUT_PATH = REPO_ROOT / "outputs/final_features/physiology_mwl_analysis_dataset.csv"
FIGURE_DIR = REPO_ROOT / "postprocessing/outputs/figures"
TABLE_DIR = REPO_ROOT / "postprocessing/outputs/tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = TABLE_DIR / "haptic_noha_physiology_by_phase.csv"
MEDIANS_PATH = TABLE_DIR / "physiology_participant_phase_medians.csv"
SUMMARY_PATH = TABLE_DIR / "haptic_noha_physiology_phase_summary.csv"
print("Input:", INPUT_PATH)

## 3. Load and validate the final physiological representation

Feature columns are identified only from the approved physiological prefixes. MWL columns are not used in this analysis.

In [ ]:
data = pd.read_csv(INPUT_PATH)
MODALITY_PREFIXES = {
    "ECG": "delta_ecg_", "EDA": "delta_eda_", "RESP": "delta_resp_",
    "TEMP": "delta_temp_", "fNIRS": "fnirs_",
}
modality_features = {m: [c for c in data.columns if c.startswith(p)] for m, p in MODALITY_PREFIXES.items()}
feature_columns = [f for m in MODALITY_ORDER for f in modality_features[m]]
feature_to_modality = {f: m for m, fs in modality_features.items() for f in fs}
assert len(feature_columns) == 79 and len(set(feature_columns)) == 79
assert set(data.phase) == set(PHASE_ORDER)
assert set(data.group) == {"Haptic", "NoHA"}
assert not data.duplicated(["participant_id", "phase", "block_index"]).any()

participant_membership = data[["participant_id", "group"]].drop_duplicates()
assert participant_membership.participant_id.nunique() == len(participant_membership)
validation = {
    "rows": len(data), "participants": data.participant_id.nunique(),
    "Haptic participants": participant_membership.group.eq("Haptic").sum(),
    "NoHA participants": participant_membership.group.eq("NoHA").sum(),
    "physiological features": len(feature_columns),
}
print(pd.Series(validation).to_string())
print("\nGroup membership:\n", participant_membership.sort_values(["group", "participant_id"]).to_string(index=False))
print("\nObservations by phase:\n", data.groupby("phase").size().reindex(PHASE_ORDER).to_string())
print("\nObservations per participant × phase:\n", data.pivot_table(index=["participant_id", "group"], columns="phase", values="block_index", aggfunc="count", fill_value=0).reindex(columns=PHASE_ORDER).to_string())
modality_missingness = pd.DataFrame([{
    "modality": m, "n_features": len(fs), "missing_cells": int(data[fs].isna().sum().sum()),
    "missing_cell_fraction": float(data[fs].isna().mean().mean()),
    "rows_entire_modality_missing": int(data[fs].isna().all(axis=1).sum()),
} for m, fs in modality_features.items()])
print("\nModality-specific missingness:\n", modality_missingness.to_string(index=False))

## 4. Participant × phase aggregation

Each physiological feature is independently summarized by the median of its available blocks. The statistical unit is therefore the participant, never an individual block.

In [ ]:
group_consistency = data.groupby("participant_id").group.nunique()
assert group_consistency.eq(1).all()
participant_phase = data.groupby(["participant_id", "group", "phase"], observed=True)[feature_columns].median().reset_index()
block_counts = data.groupby(["participant_id", "group", "phase"], observed=True).size().rename("n_blocks_total").reset_index()
participant_phase = block_counts.merge(participant_phase, on=["participant_id", "group", "phase"], validate="one_to_one")
participant_phase["phase"] = pd.Categorical(participant_phase.phase, categories=PHASE_ORDER, ordered=True)
participant_phase = participant_phase.sort_values(["participant_id", "phase"]).reset_index(drop=True)
assert not participant_phase.duplicated(["participant_id", "phase"]).any()
assert len(participant_phase) == 21 * 5
if SAVE_TABLES:
    participant_phase.to_csv(MEDIANS_PATH, index=False)
print("Participant × phase rows:", len(participant_phase))
print("Blocks per participant × phase:\n", participant_phase.groupby(["phase", "n_blocks_total"], observed=True).size().rename("participants").to_string())

## 5. Aggregation validation and missingness

In [ ]:
participant_counts = participant_phase.groupby(["phase", "group"], observed=True).size().unstack(fill_value=0).reindex(PHASE_ORDER)
missing_by_feature_phase = participant_phase.groupby("phase", observed=True)[feature_columns].apply(lambda x: x.isna().sum()).reindex(PHASE_ORDER)
missing_records = []
for phase in PHASE_ORDER:
    for feature in feature_columns:
        n_missing = int(missing_by_feature_phase.loc[phase, feature])
        if n_missing:
            unavailable = participant_phase.loc[participant_phase.phase.eq(phase) & participant_phase[feature].isna(), "participant_id"].tolist()
            missing_records.append({"phase": phase, "feature": feature, "modality": feature_to_modality[feature], "n_participants_missing": n_missing, "participants": ";".join(unavailable)})
missing_participants = pd.DataFrame(missing_records)
print("Participants per phase and group:\n", participant_counts.to_string())
print("\nFeatures with unavailable participants after aggregation:", missing_by_feature_phase.gt(0).sum(axis=1).to_dict())
if len(missing_participants):
    print("\nUnavailable participant-feature combinations:\n", missing_participants.to_string(index=False))
else:
    print("No participant-level physiological values are missing.")

## 6. Mann–Whitney and phase-specific FDR helpers

SciPy returns U for the first sample, Haptic. The primary effect size is

`rank_biserial = 2 × U_Haptic / (n_Haptic × n_NoHA) − 1`.

It ranges from −1 to +1. Positive values mean higher Haptic values; negative values mean lower Haptic values. A toy ordering assertion verifies the sign convention.

In [ ]:
toy_u = mannwhitneyu([3, 4], [1, 2], alternative="two-sided").statistic
toy_rank_biserial = 2 * toy_u / (2 * 2) - 1
assert toy_u == 4 and toy_rank_biserial == 1

def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(p.shape, np.nan)
    finite = np.isfinite(p)
    if not finite.any():
        return adjusted
    values = p[finite]
    order = np.argsort(values)
    ranked = values[order]
    m = len(ranked)
    q_ranked = np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)
    restored = np.empty(m)
    restored[order] = q_ranked
    adjusted[np.flatnonzero(finite)] = restored
    return adjusted

def compare_feature(frame, phase, feature):
    haptic = frame.loc[frame.group.eq("Haptic"), feature].dropna().astype(float)
    noha = frame.loc[frame.group.eq("NoHA"), feature].dropna().astype(float)
    base = {
        "phase": phase, "feature": feature, "modality": feature_to_modality[feature],
        "n_haptic": len(haptic), "n_noha": len(noha),
        "median_haptic": haptic.median() if len(haptic) else np.nan,
        "median_noha": noha.median() if len(noha) else np.nan,
        "iqr_haptic": iqr(haptic) if len(haptic) else np.nan,
        "iqr_noha": iqr(noha) if len(noha) else np.nan,
    }
    combined = pd.concat([haptic, noha], ignore_index=True)
    if len(haptic) < 2 or len(noha) < 2:
        return {**base, "mannwhitney_u": np.nan, "p_value": np.nan, "rank_biserial": np.nan, "status": "insufficient_data"}
    if combined.nunique() < 2:
        return {**base, "mannwhitney_u": np.nan, "p_value": np.nan, "rank_biserial": np.nan, "status": "constant_feature"}
    test = mannwhitneyu(haptic, noha, alternative="two-sided", method="auto")
    effect = 2 * float(test.statistic) / (len(haptic) * len(noha)) - 1
    return {**base, "mannwhitney_u": float(test.statistic), "p_value": float(test.pvalue), "rank_biserial": effect, "status": "ok"}

## 7. Haptic versus NoHA within each phase

In [ ]:
rows = []
for phase in PHASE_ORDER:
    phase_data = participant_phase.loc[participant_phase.phase.eq(phase)]
    phase_results = pd.DataFrame([compare_feature(phase_data, phase, feature) for feature in feature_columns])
    phase_results["p_fdr"] = benjamini_hochberg(phase_results.p_value)
    phase_results["significant_nominal"] = phase_results.p_value.lt(FDR_ALPHA)
    phase_results["significant_fdr"] = phase_results.p_fdr.lt(FDR_ALPHA)
    rows.append(phase_results)
results = pd.concat(rows, ignore_index=True)
results["phase"] = pd.Categorical(results.phase, categories=PHASE_ORDER, ordered=True)
results = results[["phase", "feature", "modality", "n_haptic", "n_noha", "median_haptic", "median_noha", "iqr_haptic", "iqr_noha", "mannwhitney_u", "p_value", "p_fdr", "rank_biserial", "significant_nominal", "significant_fdr", "status"]]
assert len(results) == 5 * 79
if SAVE_TABLES:
    results.to_csv(RESULTS_PATH, index=False)
print("Result rows:", len(results))
print(results.groupby(["phase", "status"], observed=True).size().to_string())

## 8. Phase and modality summary

Feature counts must not be interpreted as independent physiological mechanisms because many features are related or redundant.

In [ ]:
summary_rows = []
for phase in PHASE_ORDER:
    phase_result = results.loc[results.phase.eq(phase)]
    for modality in ["ALL", *MODALITY_ORDER]:
        subset = phase_result if modality == "ALL" else phase_result.loc[phase_result.modality.eq(modality)]
        summary_rows.append({
            "phase": phase, "modality": modality,
            "n_features_total": len(subset), "n_features_tested": int(subset.status.eq("ok").sum()),
            "n_nominal": int(subset.significant_nominal.sum()), "n_fdr": int(subset.significant_fdr.sum()),
        })
phase_summary = pd.DataFrame(summary_rows)
phase_summary["phase"] = pd.Categorical(phase_summary.phase, categories=PHASE_ORDER, ordered=True)
if SAVE_TABLES:
    phase_summary.to_csv(SUMMARY_PATH, index=False)
print(phase_summary.to_string(index=False))

## 9. Ranked results by phase

In [ ]:
for phase in PHASE_ORDER:
    subset = results.loc[results.phase.eq(phase)]
    columns = ["feature", "modality", "median_haptic", "median_noha", "rank_biserial", "p_value", "p_fdr", "n_haptic", "n_noha"]
    surviving = subset.loc[subset.significant_fdr, columns].sort_values("p_fdr")
    top_p = subset.loc[subset.status.eq("ok"), columns].nsmallest(TOP_N, "p_value")
    top_effect = subset.loc[subset.status.eq("ok"), columns].assign(abs_effect=lambda x: x.rank_biserial.abs()).nlargest(TOP_N, "abs_effect").drop(columns="abs_effect")
    print(f"\n=== {phase} ===")
    print("FDR-surviving features:\n", surviving.to_string(index=False) if len(surviving) else "None")
    print("\nTop 10 by raw p-value:\n", top_p.to_string(index=False))
    print("\nTop 10 by absolute rank-biserial effect:\n", top_effect.to_string(index=False))

## 10. Overview effect-size heatmap

Stars identify cells surviving the FDR correction performed separately within that phase.

In [ ]:
effect_matrix = results.pivot(index="feature", columns="phase", values="rank_biserial").reindex(index=feature_columns, columns=PHASE_ORDER)
fdr_matrix = results.pivot(index="feature", columns="phase", values="significant_fdr").reindex(index=feature_columns, columns=PHASE_ORDER)
fig, ax = plt.subplots(figsize=(10, 22))
image = ax.imshow(effect_matrix.to_numpy(dtype=float), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(PHASE_ORDER)), PHASE_ORDER, rotation=25, ha="right")
ax.set_yticks(range(len(feature_columns)), feature_columns, fontsize=5)
ax.set_title("Haptic versus NoHA rank-biserial effect size by phase\n★ phase-specific FDR q < 0.05")
for row_index in range(len(feature_columns)):
    for column_index in range(len(PHASE_ORDER)):
        if bool(fdr_matrix.iloc[row_index, column_index]):
            ax.text(column_index, row_index, "★", ha="center", va="center", fontsize=8, color="black")
boundaries = np.cumsum([len(modality_features[m]) for m in MODALITY_ORDER])[:-1]
for boundary in boundaries:
    ax.axhline(boundary - 0.5, color="black", linewidth=1.3)
starts = np.r_[0, boundaries]; ends = np.r_[boundaries, len(feature_columns)]
for modality, start, end in zip(MODALITY_ORDER, starts, ends):
    ax.text(-0.55, (start + end - 1) / 2, modality, transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=8, fontweight="bold")
colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03)
colorbar.set_label("Rank-biserial correlation\npositive = Haptic higher")
fig.subplots_adjust(left=0.43, right=0.92, top=0.96, bottom=0.05)
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "haptic_noha_physiology_effect_size_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Descriptive count summary

Counts summarize individual feature tests only; a larger count does not imply a larger overall physiological effect.

In [ ]:
overall_summary = phase_summary.loc[phase_summary.modality.eq("ALL")].set_index("phase").reindex(PHASE_ORDER)
x = np.arange(len(PHASE_ORDER)); width = 0.36
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width/2, overall_summary.n_nominal, width, label="Nominal p < 0.05")
ax.bar(x + width/2, overall_summary.n_fdr, width, label="Phase-specific FDR q < 0.05")
ax.set_xticks(x, PHASE_ORDER, rotation=25, ha="right")
ax.set(ylabel="Number of features", title="Descriptive count of Haptic–NoHA feature comparisons")
ax.legend(); ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "haptic_noha_significant_features_by_phase.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. Automated numerical report

In [ ]:
print("HAPTIC–NOHA PHYSIOLOGY COMPARISON")
for phase in PHASE_ORDER:
    phase_data = participant_phase.loc[participant_phase.phase.eq(phase)]
    phase_result = results.loc[results.phase.eq(phase)]
    total = phase_summary.loc[(phase_summary.phase.eq(phase)) & phase_summary.modality.eq("ALL")].iloc[0]
    strongest = phase_result.loc[phase_result.status.eq("ok")].assign(abs_effect=lambda x: x.rank_biserial.abs()).nlargest(5, "abs_effect")
    fdr_modalities = sorted(phase_result.loc[phase_result.significant_fdr, "modality"].unique())
    print(f"\n{phase}: Haptic={int(phase_data.group.eq('Haptic').sum())}, NoHA={int(phase_data.group.eq('NoHA').sum())}; tested={int(total.n_features_tested)}, nominal={int(total.n_nominal)}, FDR={int(total.n_fdr)}")
    print("  strongest |effects|:", "; ".join(f"{r.feature} ({r.rank_biserial:+.3f})" for _, r in strongest.iterrows()))
    print("  modalities among FDR features:", ", ".join(fdr_modalities) if fdr_modalities else "none")

print("\nQUALITATIVE NUMERICAL PATTERN (not causal interpretation)")
for phase, wording in [
    ("pre_test", "group differences before training"),
    ("test_1", "physiological differences between Haptic and NoHA during training"),
    ("test_2", "physiological differences between Haptic and NoHA during training"),
    ("test_3", "physiological differences between Haptic and NoHA during training"),
    ("evaluation", "post-training physiological differences between groups"),
]:
    n_fdr = int(overall_summary.loc[phase, "n_fdr"])
    print(f"- {phase}: {n_fdr} FDR-significant feature(s); {wording} {'observed' if n_fdr else 'not detected after FDR'}")
print("- No Group × Phase interaction was tested; significance in one phase and not another is not evidence that phases differ.")